# Phase 10 — Real HGD DataLoader Integration (`v0.10.2`)

An interactive educational walkthrough of **Phase 10: Real High-Gamma Dataset (HGD) DataLoader Integration (`v0.10.2`)** for Motor Imagery EEG Classification.
This notebook demonstrates replacing the synthetic dataset fallback in `datasets/builder.py` with the actual HGD preprocessing pipeline while preserving all higher-level training infrastructure (`Trainer`, `ExperimentRunner`, `EEGMotorImageryModel`, `ACA`, `FATE`, `EEGClassifier`).

## 1. Objective & Architectural Overview

### Target Architecture
```text
                     ┌─> Train DataLoader (80% of train1 via random split)
                     │
train1 ──> HGDDataset ──> Split Strategy
                     │
                     └─> Validation DataLoader (20% of train1)

test1  ──> HGDDataset ──> Test DataLoader (held-out test set for evaluation)
```

### Key Integration Highlights
1. **Pluggable Split Strategy**: Splits `train1` into train/validation loaders (default: 80% train / 20% validation) while keeping `test1` strictly reserved for held-out evaluation.
2. **Configurable Representation**: Supports multi-band frequency representation `(B, F, C, S)` or time-domain representation `(B, C, S)` seamlessly via configuration.
3. **Preprocessing Cache Interface**: Supports disk/memory caching (`cache.enabled: true`) to avoid redundant EEG signal preprocessing across training runs.
4. **Rich Dataset Summary**: Prints/logs detailed dataset statistics, file counts, window splits, channels, bands, and execution device.

## 2. Environment Setup & Configuration Loading

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr and not os.path.exists(os.path.join(curr, "models")):
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return curr

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from configs.config_loader import load_master_config
from datasets.path import get_dataset_root, get_train_directory, get_test_directory, validate_dataset
from datasets.loader import discover_edf_files
from datasets.pipeline import EEGPreprocessingPipeline
from datasets.dataset import HGDDataset
from datasets.builder import build_dataloaders

print(f"Project root resolved: {PROJECT_ROOT}")

## 3. Dataset Discovery & Path Resolution

In [ ]:
config = load_master_config(project_root=PROJECT_ROOT)

dataset_root = get_dataset_root(project_root=PROJECT_ROOT)
train_dir = os.path.join(dataset_root, get_train_directory(project_root=PROJECT_ROOT))
test_dir = os.path.join(dataset_root, get_test_directory(project_root=PROJECT_ROOT))

print(f"Dataset Root: {dataset_root}")
print(f"Train Directory: {train_dir}")
print(f"Test Directory: {test_dir}")

train_files = discover_edf_files(train_dir)
test_files = discover_edf_files(test_dir)

print(f"Discovered {len(train_files)} train EDF files.")
print(f"Discovered {len(test_files)} test EDF files.")

## 4. Building DataLoaders with `build_dataloaders`

We construct `train_loader`, `val_loader`, and `test_loader` using `build_dataloaders(config)`. The dataset summary table is automatically emitted.

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(config, project_root=PROJECT_ROOT)

print(f"Train DataLoader batches : {len(train_loader)}")
print(f"Val DataLoader batches   : {len(val_loader)}")
print(f"Test DataLoader batches  : {len(test_loader)}")

## 5. Batch Inspection & Class Distribution Analysis

In [ ]:
batch_x, batch_y = next(iter(train_loader))

print(f"Sample Batch X Tensor Shape : {batch_x.shape}")
print(f"Sample Batch y Label Shape  : {batch_y.shape}")
print(f"Batch Data Type             : {batch_x.dtype}")
print(f"Batch Mean                  : {batch_x.mean().item():.4f}")
print(f"Batch Std Dev               : {batch_x.std().item():.4f}")
print(f"Sample Class Labels         : {batch_y.tolist()}")

## 6. Multi-Band Feature Visualization

Visualizing sample window multi-band spectral profiles across Theta, Alpha, Beta, and Gamma bands.

In [ ]:
if batch_x.ndim == 4:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    band_names = ["Theta (4-8 Hz)", "Alpha (8-13 Hz)", "Beta (13-30 Hz)", "Gamma (30-38 Hz)"]
    
    for i, ax in enumerate(axes.flat):
        if i < batch_x.shape[1]:
            im = ax.imshow(batch_x[0, i].numpy(), aspect="auto", cmap="viridis")
            ax.set_title(band_names[i])
            ax.set_xlabel("Time Samples (250 Hz)")
            ax.set_ylabel("EEG Channels (133)")
            fig.colorbar(im, ax=ax)
            
    plt.suptitle("Sample EEG Window Multi-Band Spectral Power Profiles", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 4))
    plt.plot(batch_x[0, 0].numpy(), label="Channel 0")
    plt.plot(batch_x[0, 1].numpy(), label="Channel 1")
    plt.title("Time-Domain EEG Window Signals")
    plt.xlabel("Samples")
    plt.ylabel("Normalized Amplitude")
    plt.legend()
    plt.show()

## 7. Conclusion

Phase 10 Patch `v0.10.2` successfully connects higher-level training infrastructure (`Trainer`, `ExperimentRunner`) directly to the real High-Gamma Dataset (HGD) preprocessing pipeline.

- Default synthetic dataset fallback has been replaced with real dataset loading.
- `train1` split into train and validation DataLoaders deterministically.
- `test1` preserved strictly as a held-out test DataLoader.
- Dataset caching interface added to optimize processing times.
- Full public API compatibility (`train_loader, val_loader, test_loader = build_dataloaders(config)`) maintained.